In [1]:
# -------------------------
# BLOCK W2-1 — SETUP
# -------------------------
# Woche 2: Faithfulness / Entailment Evaluation
# Wir verwenden dasselbe RAG-System wie in Woche 1,
# bauen es aber NICHT neu auf.

import re
import torch
from datasets import load_dataset

print("CUDA available:", torch.cuda.is_available())


C:\HHZ\KI\envs\ragthesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: False


In [2]:
# -------------------------
# BLOCK W2-1.1 — DATASET (STREAMING)
# -------------------------

import pickle
import os

# Check if local file exists
if os.path.exists("nq_3000.pkl"):
    print("Loading from local file...")
    with open("nq_3000.pkl", "rb") as f:
        data_3000 = pickle.load(f)
    print(f"Loaded {len(data_3000)} examples from local file")
else:
    print("Loading from HuggingFace (first time)...")
    ds_stream = load_dataset("natural_questions", split="train", streaming=True)
    data_3000 = list(islice(ds_stream, 3000))
    
    # Save for next time
    print("Saving locally for future runs...")
    with open("nq_3000.pkl", "wb") as f:
        pickle.dump(data_3000, f)
    print("Saved!")


Loading from local file...
Loaded 3000 examples from local file


In [3]:
# -------------------------
# BLOCK W2-1.2 — HELPERS
# -------------------------

def normalize(text):
    if text is None:
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_question(ex):
    return ex.get("question", {}).get("text", "")


In [4]:
# -------------------------
# BLOCK W2-1.3 — GOLD ANSWERS
# -------------------------

def get_gold_answer(ex):
    ann = ex.get("annotations", None)
    if not isinstance(ann, dict):
        return None

    toks = ex.get("document", {}).get("tokens", {}).get("token", [])

    sa_list = ann.get("short_answers", [])
    if not isinstance(sa_list, list) or len(sa_list) == 0:
        return None

    sa0 = sa_list[0]
    if not isinstance(sa0, dict):
        return None

    txt = sa0.get("text")
    if isinstance(txt, list) and len(txt) > 0 and txt[0]:
        return txt[0]
    if isinstance(txt, str) and txt:
        return txt

    st = sa0.get("start_token")
    en = sa0.get("end_token")

    if isinstance(st, list):
        st = st[0] if st else None
    if isinstance(en, list):
        en = en[0] if en else None

    try:
        st = int(st)
        en = int(en)
    except:
        return None

    if toks and en > st:
        return " ".join(toks[st:en]).strip()

    return None


golds = [get_gold_answer(ex) for ex in data_3000]
covered_idx = [i for i, g in enumerate(golds) if g is not None]

print("Coverage:", len(covered_idx), "/", len(data_3000))


Coverage: 1035 / 3000


In [5]:
# -------------------------
# BLOCK W2-0 — RAG SYSTEM (DPR VERSION)
# -------------------------
# Same structure as your BM25 block, but Retriever is now DPR (DensePassageRetriever)
# using a FAISSDocumentStore. We MUST compute embeddings once via update_embeddings().

import os
import shutil
import torch

from haystack import Document
from haystack.document_stores import FAISSDocumentStore
from haystack.nodes import DensePassageRetriever

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------
# 0) (Optional) Clean rebuild switch
# -------------------------
# If you get weird FAISS errors later (vector_id mismatch, reconstruct error),
# set CLEAN_BUILD=True once, run the cell, then set it back to False.
CLEAN_BUILD = True

BASE_DIR = "haystack_dpr_t5_store_w2"
if CLEAN_BUILD and os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

# -------------------------
# 1) Build Document Store (FAISS)
# -------------------------
# IMPORTANT: When creating a NEW store, do NOT pass faiss_index_path/config_path.
# Those are only for loading an existing saved store.
doc_store = FAISSDocumentStore(
    sql_url=f"sqlite:///{BASE_DIR}/dpr_docs.db",
    faiss_index_factory_str="Flat",
    similarity="dot_product",
    return_embedding=True,
    validate_index_sync=False
)

# -------------------------
# 2) Passage extraction (unchanged)
# -------------------------
def extract_passages(ex, max_passages=2):
    toks = ex.get("document", {}).get("tokens", {}).get("token", [])
    lac = ex.get("long_answer_candidates", {})
    starts = lac.get("start_token", [])
    ends = lac.get("end_token", [])

    passages = []
    for s, e in zip(starts, ends):
        if s is not None and e is not None and e > s:
            txt = " ".join(toks[s:e]).strip()
            if txt:
                passages.append(txt)
        if len(passages) >= max_passages:
            break
    return passages

# -------------------------
# 3) Build docs (unchanged)
# -------------------------
docs = []
for i, ex in enumerate(data_3000):
    for j, p in enumerate(extract_passages(ex)):
        docs.append(Document(content=p, meta={"ex_id": i, "p_id": j}))

doc_store.write_documents(docs)

# -------------------------
# 4) DPR Retriever + embeddings
# -------------------------
DPR_QUERY = "facebook/dpr-question_encoder-single-nq-base"
DPR_CTX   = "facebook/dpr-ctx_encoder-single-nq-base"

retriever = DensePassageRetriever(
    document_store=doc_store,
    query_embedding_model=DPR_QUERY,
    passage_embedding_model=DPR_CTX,
    max_seq_len_query=256,
    max_seq_len_passage=256,
    batch_size=16,
    use_gpu=torch.cuda.is_available(),
    embed_title=False
)

# DPR NEEDS embeddings in FAISS:
doc_store.update_embeddings(retriever, batch_size=16)

# -------------------------
# 5) Generator (unchanged)
# -------------------------
GEN_MODEL = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(device)
gen_model.eval()

# -------------------------
# 6) RAG Answer Functions (same interface)
# -------------------------
def rag_answer(question, top_k=10, max_new_tokens=64):
    retrieved = retriever.retrieve(question, top_k=top_k)
    context = " ".join([d.content for d in retrieved])

    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not contained in the context, say 'I don't know'.\n\n"
        f"Context:\n{context}\n\nQuestion:\n{question}\nAnswer:"
    )

    inputs = gen_tok(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    answer = gen_tok.decode(out[0], skip_special_tokens=True)
    return answer, retrieved, context


# W2-0B — RAG VARIANT B (ANSWER-SEEKING) (unchanged logic)
PROMPT_B = """Answer the question as best as you can using the context below.
If the answer is not explicitly stated, make the best possible inference.

Context:
{context}

Question:
{question}

Answer:"""

def rag_answer_B(question, top_k=10, max_new_tokens=64):
    retrieved = retriever.retrieve(question, top_k=top_k)
    context = " ".join([d.content for d in retrieved])

    prompt = PROMPT_B.format(context=context, question=question)

    inputs = gen_tok(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    answer = gen_tok.decode(out[0], skip_special_tokens=True)
    return answer, retrieved, context

print("RAG system ready (DPR + FAISS + T5). Docs:", doc_store.get_document_count())


C:\HHZ\KI\envs\ragthesis\Lib\site-packages\quantulum3\classifier.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Writing Documents: 10000it [00:22, 446.60it/s]                                                                         
Create embeddings: 100%|████████████████████████████████████████████████████████████| 16/16 [00:01<00:00,  8.73 Docs/s]
Documents Processed: 5440 docs [25:05,  3.61 docs/s]                                                                   


RAG system ready (DPR + FAISS + T5). Docs: 5431


In [6]:
# -------------------------
# BLOCK W2-3 — SIMPLE FAITHFULNESS (BASELINE)
# -------------------------
# Faithfulness-Support = 1, wenn Answer im Kontext vorkommt (nach Normalisierung)

def is_supported_by_context(answer, context):
    if answer is None or context is None:
        return None
    a = normalize(answer)
    c = normalize(context)
    if not a or not c:
        return None
    return int(a in c)



1
0


In [7]:
# -------------------------
# BLOCK W2-4 — NLI MODEL (ENTAILMENT) FOR FAITHFULNESS
# -------------------------
# Wir verwenden ein MNLI (entailment) Modell:
# premise = Kontext
# hypothesis = Antwort
# Output: entailment / neutral / contradiction

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

NLI_MODEL = "roberta-large-mnli"  # sehr gut, ggf. groß
# Alternative falls zu langsam/VRAM: "roberta-base-mnli" oder "microsoft/deberta-base-mnli"

device = "cuda" if torch.cuda.is_available() else "cpu"

nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
nli_model.eval()

print("Loaded NLI model:", NLI_MODEL, "on", device)


Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loaded NLI model: roberta-large-mnli on cpu


In [8]:
# -------------------------
# BLOCK W2-4.1 — ENTAILMENT SCORE FUNCTION
# -------------------------

def entailment_label_and_score(premise, hypothesis, max_length=256):
    """
    premise: Kontext
    hypothesis: Antwort
    Return: (label, entailment_prob)
    """
    if premise is None or hypothesis is None:
        return None, None

    inputs = nli_tok(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze().detach().cpu().numpy()

    # MNLI labels: usually [contradiction, neutral, entailment]
    labels = ["contradiction", "neutral", "entailment"]
    pred_id = int(probs.argmax())
    return labels[pred_id], float(probs[2])  # entailment prob = index 2


In [9]:
# -------------------------
# BLOCK W2-5.0 — METRIC HELPERS (EM / F1 / HIT@K)
# -------------------------

def exact_match(pred, gold):
    if gold is None:
        return None
    return int(normalize(pred) == normalize(gold))

def f1_score(pred, gold):
    if gold is None:
        return None
    pred_toks = normalize(pred).split()
    gold_toks = normalize(gold).split()

    if len(pred_toks) == 0 and len(gold_toks) == 0:
        return 1.0
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0

    # multiset overlap
    counts = {}
    for t in pred_toks:
        counts[t] = counts.get(t, 0) + 1

    num_same = 0
    for t in gold_toks:
        if counts.get(t, 0) > 0:
            num_same += 1
            counts[t] -= 1

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

def hit_at_k(gold, retrieved_docs):
    """Proxy Hit@k: gold answer appears somewhere in concatenated retrieved docs."""
    if gold is None:
        return None
    g = normalize(gold)
    if not g:
        return None
    ctx = "\n".join([d.content for d in retrieved_docs])
    return int(g in normalize(ctx))


In [12]:
# =================================================================
# BLOCK W2-6 — FULL RUN (A vs B) + SAVE (UPDATED FOR 3000 SAMPLES)
# =================================================================

import pandas as pd

k = 10
rows = []

print(f"Starting full evaluation on {len(data_3000)} samples...")

for idx, i in enumerate(range(len(data_3000))):  # ← ALLE 3000!
    # Progress tracking
    if idx % 100 == 0:
        print(f"Progress: {idx}/{len(data_3000)} ({idx/len(data_3000)*100:.1f}%)")
    
    # Get sample
    ex = data_3000[i]  # ← data_3000 statt data_1500
    q = extract_question(ex)
    gold = get_gold_answer(ex)  # Kann None sein!
    
    # ========== VARIANT A (Conservative) ==========
    pred_A, docs_A, context_A = rag_answer(q, top_k=k)
    ent_label_A, ent_score_A = entailment_label_and_score(context_A, pred_A)
    ent_faith_A = ent_faith_binary(ent_label_A, ent_score_A)
    
    # ========== VARIANT B (Answer-Seeking) ==========
    pred_B, docs_B, context_B = rag_answer_B(q, top_k=k)
    ent_label_B, ent_score_B = entailment_label_and_score(context_B, pred_B)
    ent_faith_B = ent_faith_binary(ent_label_B, ent_score_B)
    
    # ========== COLLECT RESULTS ==========
    rows.append({
        "idx": i,
        "question": q,
        "gold": gold,
        "has_gold": gold is not None,  # ← NEU: Flag!
        
        # Variant A
        "pred_A": pred_A,
        "em_A": exact_match(pred_A, gold) if gold else None,
        "f1_A": f1_score(pred_A, gold) if gold else None,
        f"hit@{k}_A": hit_at_k(gold, docs_A) if gold else None,
        "ent_label_A": ent_label_A,
        "ent_faith_A": ent_faith_A,
        "ent_score_A": ent_score_A,
        
        # Variant B
        "pred_B": pred_B,
        "em_B": exact_match(pred_B, gold) if gold else None,
        "f1_B": f1_score(pred_B, gold) if gold else None,
        f"hit@{k}_B": hit_at_k(gold, docs_B) if gold else None,
        "ent_label_B": ent_label_B,
        "ent_faith_B": ent_faith_B,
        "ent_score_B": ent_score_B,
    })

# Create DataFrame
df_w2_full = pd.DataFrame(rows)

print(f"\n Evaluation complete!")
print(f"Total samples: {len(df_w2_full)}")
print(f"Samples with gold: {df_w2_full['has_gold'].sum()}")

# ========== SAVE RESULTS ==========

# Save ALL samples
csv_all = "02_results_dpr_T5_faithfulness_ALL_3000.csv"
df_w2_full.to_csv(csv_all, index=False)
print(f" Saved: {csv_all}")

# Save Gold subset separately
df_gold = df_w2_full[df_w2_full['has_gold'] == True]
csv_gold = "02_results_dpr_T5_faithfulness_GOLD_ONLY_3000.csv"
df_gold.to_csv(csv_gold, index=False)
print(f" Saved: {csv_gold}")

# Quick preview
print(f"\n First 5 rows:")
print(df_w2_full.head())

Starting full evaluation on 3000 samples...
This will take ~4-5 hours. Progress updates every 100 samples.
Progress: 0/3000 (0.0%)
Progress: 100/3000 (3.3%)
Progress: 200/3000 (6.7%)
Progress: 300/3000 (10.0%)
Progress: 400/3000 (13.3%)
Progress: 500/3000 (16.7%)
Progress: 600/3000 (20.0%)
Progress: 700/3000 (23.3%)
Progress: 800/3000 (26.7%)
Progress: 900/3000 (30.0%)
Progress: 1000/3000 (33.3%)
Progress: 1100/3000 (36.7%)
Progress: 1200/3000 (40.0%)
Progress: 1300/3000 (43.3%)
Progress: 1400/3000 (46.7%)
Progress: 1500/3000 (50.0%)
Progress: 1600/3000 (53.3%)
Progress: 1700/3000 (56.7%)
Progress: 1800/3000 (60.0%)
Progress: 1900/3000 (63.3%)
Progress: 2000/3000 (66.7%)
Progress: 2100/3000 (70.0%)
Progress: 2200/3000 (73.3%)
Progress: 2300/3000 (76.7%)
Progress: 2400/3000 (80.0%)
Progress: 2500/3000 (83.3%)
Progress: 2600/3000 (86.7%)
Progress: 2700/3000 (90.0%)
Progress: 2800/3000 (93.3%)
Progress: 2900/3000 (96.7%)

✅ Evaluation complete!
Total samples: 3000
Samples with gold: 1035


In [13]:
# =================================================================
# BLOCK W2-7 — SUMMARY (A vs B) + SAVE (UPDATED FOR 3000 SAMPLES)
# =================================================================

import pandas as pd
import numpy as np

# Erwartet: df_w2_full existiert aus BLOCK W2-6
# Falls k hier nicht mehr im Scope ist, setze ihn nochmal:
try:
    k
except NameError:
    k = 10

def safe_mean(series: pd.Series):
    """Mean über numerische Werte, ignoriert None/NaN."""
    s = pd.to_numeric(series, errors="coerce")
    return float(s.dropna().mean()) if len(s.dropna()) else np.nan

def safe_rate(series: pd.Series):
    """Rate für binäre Spalten (0/1), ignoriert None/NaN."""
    s = pd.to_numeric(series, errors="coerce")
    return float(s.dropna().mean()) if len(s.dropna()) else np.nan

# Spaltennamen dynamisch anhand k
hitA_col = f"hit@{k}_A"
hitB_col = f"hit@{k}_B"

# Sanity-check: welche Spalten fehlen?
expected_cols = [
    "has_gold",  # ← NEU!
    "em_A", "f1_A", hitA_col, "ent_faith_A", "ent_score_A",
    "em_B", "f1_B", hitB_col, "ent_faith_B", "ent_score_B",
]
missing = [c for c in expected_cols if c not in df_w2_full.columns]
if missing:
    print(" Missing columns in df_w2_full:", missing)
    print("Available columns:", list(df_w2_full.columns))
    raise KeyError("df_w2_full does not contain required columns for summary.")

# Split: ALL samples vs Gold-only
n_all = len(df_w2_full)
df_gold_only = df_w2_full[df_w2_full['has_gold'] == True]
n_gold = len(df_gold_only)

print("="*70)
print(" SUMMARY STATISTICS")
print("="*70)
print(f"\nTotal samples: {n_all}")
print(f"Samples with gold: {n_gold}")
print(f"Samples without gold: {n_all - n_gold}")

# ========== SUMMARY FOR ALL SAMPLES (Faithfulness only) ==========
print("\n" + "="*70)
print(" FAITHFULNESS METRICS (ALL 3000 SAMPLES)")
print("="*70)

summary_all_A = {
    "variant": "A (Conservative)",
    "dataset": "All 3000",
    "n_eval": n_all,
    "entailment_faithfulness_rate": safe_rate(df_w2_full["ent_faith_A"]),
    "entailment_score_mean": safe_mean(df_w2_full["ent_score_A"]),
}

summary_all_B = {
    "variant": "B (Answer-Seeking)",
    "dataset": "All 3000",
    "n_eval": n_all,
    "entailment_faithfulness_rate": safe_rate(df_w2_full["ent_faith_B"]),
    "entailment_score_mean": safe_mean(df_w2_full["ent_score_B"]),
}

df_summary_all = pd.DataFrame([summary_all_A, summary_all_B])
print(df_summary_all.to_string(index=False))

# ========== SUMMARY FOR GOLD SUBSET (Correctness + Faithfulness) ==========
print("\n" + "="*70)
print(f" CORRECTNESS + FAITHFULNESS (GOLD SUBSET, N={n_gold})")
print("="*70)

summary_gold_A = {
    "variant": "A (Conservative)",
    "dataset": f"Gold subset ({n_gold})",
    "n_eval": n_gold,
    "em_mean": safe_mean(df_gold_only["em_A"]),
    "f1_mean": safe_mean(df_gold_only["f1_A"]),
    f"hit@{k}_rate": safe_rate(df_gold_only[hitA_col]),
    "entailment_faithfulness_rate": safe_rate(df_gold_only["ent_faith_A"]),
    "entailment_score_mean": safe_mean(df_gold_only["ent_score_A"]),
}

summary_gold_B = {
    "variant": "B (Answer-Seeking)",
    "dataset": f"Gold subset ({n_gold})",
    "n_eval": n_gold,
    "em_mean": safe_mean(df_gold_only["em_B"]),
    "f1_mean": safe_mean(df_gold_only["f1_B"]),
    f"hit@{k}_rate": safe_rate(df_gold_only[hitB_col]),
    "entailment_faithfulness_rate": safe_rate(df_gold_only["ent_faith_B"]),
    "entailment_score_mean": safe_mean(df_gold_only["ent_score_B"]),
}

df_summary_gold = pd.DataFrame([summary_gold_A, summary_gold_B])
print(df_summary_gold.to_string(index=False))

# ========== COMPARISON A vs B ==========
print("\n" + "="*70)
print(" COMPARISON: VARIANT A vs B")
print("="*70)

# All samples comparison
faith_delta_all = (summary_all_A["entailment_faithfulness_rate"] - 
                   summary_all_B["entailment_faithfulness_rate"])
print(f"\nFaithfulness Δ (A - B, all samples): {faith_delta_all:+.3f}")

# Gold subset comparison
em_delta = summary_gold_A["em_mean"] - summary_gold_B["em_mean"]
f1_delta = summary_gold_A["f1_mean"] - summary_gold_B["f1_mean"]
faith_delta_gold = (summary_gold_A["entailment_faithfulness_rate"] - 
                    summary_gold_B["entailment_faithfulness_rate"])

print(f"\nGold subset deltas (A - B):")
print(f"  EM Δ: {em_delta:+.3f}")
print(f"  F1 Δ: {f1_delta:+.3f}")
print(f"  Faithfulness Δ: {faith_delta_gold:+.3f}")

# ========== COVERAGE ANALYSIS ==========
print("\n" + "="*70)
print(" COVERAGE ANALYSIS (I don't know rate)")
print("="*70)

def count_idk(df, pred_col):
    """Count how often prediction contains 'I don't know' or similar."""
    idk_patterns = ["don't know", "do not know", "i don't", "cannot answer", "no answer"]
    count = 0
    for pred in df[pred_col]:
        if pred and isinstance(pred, str):
            pred_lower = pred.lower()
            if any(pattern in pred_lower for pattern in idk_patterns):
                count += 1
    return count

idk_A = count_idk(df_w2_full, "pred_A")
idk_B = count_idk(df_w2_full, "pred_B")

print(f"\nVariant A 'I don't know' count: {idk_A} ({idk_A/n_all*100:.1f}%)")
print(f"Variant B 'I don't know' count: {idk_B} ({idk_B/n_all*100:.1f}%)")
print(f"Coverage difference (A - B): {(idk_A - idk_B)/n_all*100:+.1f}%")

# ========== SAVE SUMMARY ==========

# Combine summaries
df_summary_combined = pd.concat([
    df_summary_all.assign(subset="All samples"),
    df_summary_gold.assign(subset="Gold only")
], ignore_index=True)

# Reorder columns
cols = ["variant", "subset", "n_eval", "em_mean", "f1_mean", 
        f"hit@{k}_rate", "entailment_faithfulness_rate", "entailment_score_mean"]
df_summary_combined = df_summary_combined[[c for c in cols if c in df_summary_combined.columns]]

# Save
csv_summary = "02_results_dpr_T5_faithfulness_SUMMARY_3000.csv"
df_summary_combined.to_csv(csv_summary, index=False)
print(f"\n Saved summary: {csv_summary}")

# Display
print("\n" + "="*70)
print(" FINAL SUMMARY TABLE")
print("="*70)
print(df_summary_combined.to_string(index=False))

# ========== ADDITIONAL ANALYSIS: GOLD vs NON-GOLD FAITHFULNESS ==========
print("\n" + "="*70)
print(" FAITHFULNESS: GOLD vs NON-GOLD COMPARISON")
print("="*70)

df_non_gold = df_w2_full[df_w2_full['has_gold'] == False]

print("\nVariant A:")
print(f"  Faithfulness (with gold):    {safe_rate(df_gold_only['ent_faith_A']):.3f}")
print(f"  Faithfulness (without gold): {safe_rate(df_non_gold['ent_faith_A']):.3f}")

print("\nVariant B:")
print(f"  Faithfulness (with gold):    {safe_rate(df_gold_only['ent_faith_B']):.3f}")
print(f"  Faithfulness (without gold): {safe_rate(df_non_gold['ent_faith_B']):.3f}")

print("\n" + "="*70)
print(" ALL ANALYSIS COMPLETE!")
print("="*70)

📊 SUMMARY STATISTICS

Total samples: 3000
Samples with gold: 1035
Samples without gold: 1965

📈 FAITHFULNESS METRICS (ALL 3000 SAMPLES)
           variant  dataset  n_eval  entailment_faithfulness_rate  entailment_score_mean
  A (Conservative) All 3000    3000                         0.073               0.173787
B (Answer-Seeking) All 3000    3000                         0.403               0.380303

📈 CORRECTNESS + FAITHFULNESS (GOLD SUBSET, N=1035)
           variant            dataset  n_eval  em_mean  f1_mean  hit@10_rate  entailment_faithfulness_rate  entailment_score_mean
  A (Conservative) Gold subset (1035)    1035 0.000000 0.001347      0.34686                      0.066667               0.169266
B (Answer-Seeking) Gold subset (1035)    1035 0.001932 0.030234      0.34686                      0.397101               0.372199

🔄 COMPARISON: VARIANT A vs B

Faithfulness Δ (A - B, all samples): -0.330

Gold subset deltas (A - B):
  EM Δ: -0.002
  F1 Δ: -0.029
  Faithfulness Δ: -0.